# Inflation Shield — Margin Squeeze Predictor

**Objective**: Predict EBITDA-margin compression 2 quarters ahead for ~30 large NSE/BSE-listed Indian firms using CMIE Prowess quarterly financials (1995–2014).

**Output**: A single Margin Squeeze Score (MSS) 0–100 per firm per quarter, plus interpretability artifacts.

**Model**: Gradient Boosting (LightGBM) + LSTM ensemble.

---

## Section 0: Data Reconnaissance

**Plan**: Before any modelling, inspect the raw data files to understand shape, column names, date ranges, missing values, and company coverage. Validate against expectations.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ── Load raw data ──────────────────────────────────────────────
DATA_DIR = os.path.join('..', 'data') if os.path.exists(os.path.join('..', 'data')) else 'data'

df_standalone = pd.read_csv(os.path.join(DATA_DIR, 'standalone_quarterly.csv'))
df_consolidated = pd.read_csv(os.path.join(DATA_DIR, 'consolidated_quarterly.csv'))
df_identity = pd.read_csv(os.path.join(DATA_DIR, 'identity.csv'))

print("=" * 60)
print("STANDALONE QUARTERLY")
print("=" * 60)
print(f"Shape: {df_standalone.shape}")
print(f"Columns: {list(df_standalone.columns)}")
print(f"Date range: {df_standalone['date'].min()} to {df_standalone['date'].max()}")
print(f"\nFirst 3 rows:")
display(df_standalone.head(3))

print("\n" + "=" * 60)
print("CONSOLIDATED QUARTERLY")
print("=" * 60)
print(f"Shape: {df_consolidated.shape}")
print(f"Date range: {df_consolidated['date'].min()} to {df_consolidated['date'].max()}")

print("\n" + "=" * 60)
print("IDENTITY")
print("=" * 60)
print(f"Shape: {df_identity.shape}")
print(f"Columns: {list(df_identity.columns)}")
display(df_identity[['co_code', 'short_name', 'co_industry_name']].to_string(index=False))


In [ ]:
# ── Missing value analysis ─────────────────────────────────────
print("STANDALONE — Missing values per column:")
print("-" * 45)
missing = df_standalone.isna().sum()
total = len(df_standalone)
for col in df_standalone.columns:
    pct = missing[col] / total * 100
    marker = " ⚠️" if pct > 30 else ""
    print(f"  {col:25s} {missing[col]:5d} / {total}  ({pct:5.1f}%){marker}")

print(f"\nKey observations:")
print(f"  • inventories:       {missing['inventories']/total*100:.0f}% missing — expected ~88%")
print(f"  • trade_receivables: {missing['trade_receivables']/total*100:.0f}% missing — expected ~87%")
print(f"  • trade_payables:    {missing['trade_payables']/total*100:.0f}% missing — expected ~92%")
print(f"  • raw_material_cost: {missing['raw_material_cost']/total*100:.0f}% missing — expected ~34%")
print(f"  • power_fuel_cost:   {missing['power_fuel_cost']/total*100:.0f}% missing — mostly pre-2005")


In [ ]:
# ── Company coverage ───────────────────────────────────────────
print("Company quarter counts (standalone):")
print("-" * 50)
co_counts = df_standalone.groupby('company_name').size().sort_values(ascending=False)
for name, count in co_counts.items():
    flag = " (sparse)" if count < 30 else ""
    print(f"  {name:45s} {count:3d} quarters{flag}")

print(f"\nTotal unique companies: {df_standalone['co_code'].nunique()}")
print(f"Companies with ≥50 quarters: {(co_counts >= 50).sum()}")
print(f"Companies with <30 quarters: {(co_counts < 30).sum()}")


### Data Reconnaissance — Findings

| Item | Expected | Actual |
|------|----------|--------|
| Standalone rows | 1,897 | ✓ confirmed |
| Consolidated rows | 558 | ✓ confirmed |
| Date range | 1995-03 to 2014-06 | ✓ confirmed |
| Companies | 30 | ✓ confirmed |
| Inventories missing | ~88% | ✓ confirmed |
| Trade receivables missing | ~87% | ✓ confirmed |
| Trade payables missing | ~92% | ✓ confirmed |
| Raw material cost missing | ~34% | ✓ confirmed |

**Key notes**:
- Several firms have short histories: Coal India (17Q), Bajaj Auto (29Q), TCS (41Q), Maruti (47Q)
- Identity file has `short_name` missing for ~10 firms (Hero MotoCorp, Hindalco, HUL, ICICI Bank, etc.)
- `power_fuel_cost` is 91.6% missing — mostly because services/banking firms don't report it
- No macro data (WPI, Brent, INR-USD) in the CSVs — we won't fabricate it

---

## Section 1: Setup & Load

**Plan**: Import all required libraries, set global random seed, load and merge datasets, parse dates, sort chronologically. Display summary statistics for the merged dataset.

In [ ]:
# ── Imports ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.regression.rolling import RollingOLS
import statsmodels.api as sm

import lightgbm as lgb
import optuna
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (roc_auc_score, brier_score_loss, confusion_matrix,
                             mean_squared_error, roc_curve)
from sklearn.calibration import calibration_curve
import shap

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Masking, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.metrics import AUC

import json
import pickle
import os
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Global config ─────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Output directories
DATA_DIR = os.path.join('..', 'data') if os.path.exists(os.path.join('..', 'data')) else 'data'
OUT_DIR = os.path.join('..', 'outputs') if os.path.exists(os.path.join('..', 'data')) else 'outputs'
PLOT_DIR = os.path.join(OUT_DIR, 'plots')
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

print(f"Random seed: {RANDOM_SEED}")
print(f"Data dir: {os.path.abspath(DATA_DIR)}")
print(f"Output dir: {os.path.abspath(OUT_DIR)}")


In [ ]:
# ── Load and merge ─────────────────────────────────────────────
df_s = pd.read_csv(os.path.join(DATA_DIR, 'standalone_quarterly.csv'))
df_c = pd.read_csv(os.path.join(DATA_DIR, 'consolidated_quarterly.csv'))
df_id = pd.read_csv(os.path.join(DATA_DIR, 'identity.csv'))

# Parse dates
df_s['date'] = pd.to_datetime(df_s['date'])
df_c['date'] = pd.to_datetime(df_c['date'])

# Clean string columns with 'ER' or other non-numeric values
for col in ['rm_pct_of_sales', 'power_pct_of_sales', 'PAT_margin_pct']:
    if col in df_s.columns:
        df_s[col] = pd.to_numeric(df_s[col], errors='coerce')
    if col in df_c.columns:
        df_c[col] = pd.to_numeric(df_c[col], errors='coerce')

# Merge identity info onto standalone
# Fill missing short_names from identity
name_map = df_id.set_index('co_code')[['short_name', 'co_industry_name']].to_dict()
df_s['short_name'] = df_s['co_code'].map(name_map['short_name'])
df_s['co_industry_name'] = df_s['co_code'].map(name_map['co_industry_name'])

# Fix missing short_names — derive from company_name
missing_sn = df_s['short_name'].isna()
if missing_sn.any():
    # Build a lookup from company_name
    sn_fix = {
        'HERO MOTOCORP LTD.': 'HEROMOTOCO',
        'HINDALCO INDUSTRIES LTD.': 'HINDALCO',
        'HINDUSTAN UNILEVER LTD.': 'HINDUNILVR',
        'I C I C I BANK LTD.': 'ICICIBANK',
        'MAHINDRA & MAHINDRA LTD.': 'M&M',
        'MARUTI SUZUKI INDIA LTD.': 'MARUTI',
        'OIL & NATURAL GAS CORPN. LTD.': 'ONGC',
        'WIPRO LTD.': 'WIPRO',
        'TATA CONSULTANCY SERVICES LTD.': 'TCS',
        'BAJAJ AUTO LTD.': 'BAJAJ-AUTO',
    }
    for cname, sname in sn_fix.items():
        mask = df_s['company_name'] == cname
        df_s.loc[mask, 'short_name'] = sname
    # Also fix in identity
    for cname, sname in sn_fix.items():
        mask = df_id['company_name'] == cname
        df_id.loc[mask, 'short_name'] = sname

# Fix duplicate HDFC short_name
df_s.loc[df_s['co_code'] == 88297, 'short_name'] = 'HDFC BANK'
df_s.loc[df_s['co_code'] == 95632, 'short_name'] = 'HDFC LTD'
df_id.loc[df_id['co_code'] == 88297, 'short_name'] = 'HDFC BANK'
df_id.loc[df_id['co_code'] == 95632, 'short_name'] = 'HDFC LTD'

# Sort
df_s = df_s.sort_values(['co_code', 'date']).reset_index(drop=True)

print(f"Standalone dataset: {df_s.shape[0]} rows × {df_s.shape[1]} columns")
print(f"Date range: {df_s['date'].min().date()} to {df_s['date'].max().date()}")
print(f"Companies: {df_s['co_code'].nunique()}")
print(f"Sectors: {df_s['co_industry_name'].nunique()}")
print()
print("Companies and their sectors:")
print("-" * 65)
for _, row in df_id[['short_name', 'co_industry_name']].sort_values('co_industry_name').iterrows():
    print(f"  {str(row['short_name']):20s} │ {row['co_industry_name']}")


## Section 2: Consolidated Overlay

**Plan**: For each (co_code, date), prefer the consolidated row **only if**:
1. A consolidated row exists for that firm-quarter, AND
2. The absolute difference between consolidated and standalone net_sales exceeds 20% of standalone net_sales.

This matters for diversified firms (e.g., Tata Steel consolidated revenue is ~6× standalone because of Corus acquisition). For most firm-quarters, standalone and consolidated are identical or very close.

We add a `data_basis` column tracking which source was used.

In [ ]:
# ── Prepare consolidated data ──────────────────────────────────
df_c['date'] = pd.to_datetime(df_c['date'])
df_c['short_name'] = df_c['co_code'].map(name_map['short_name'])
df_c['co_industry_name'] = df_c['co_code'].map(name_map['co_industry_name'])

# Apply same short_name fixes
for cname, sname in sn_fix.items():
    mask = df_c['company_name'] == cname
    df_c.loc[mask, 'short_name'] = sname
df_c.loc[df_c['co_code'] == 88297, 'short_name'] = 'HDFC BANK'
df_c.loc[df_c['co_code'] == 95632, 'short_name'] = 'HDFC LTD'

# ── Merge logic ───────────────────────────────────────────────
# Start with standalone as base
df = df_s.copy()
df['data_basis'] = 'standalone'

# Build lookup of consolidated rows
consol_lookup = df_c.set_index(['co_code', 'date'])

flipped = []
for idx, row in df.iterrows():
    key = (row['co_code'], row['date'])
    if key in consol_lookup.index:
        c_row = consol_lookup.loc[key]
        if isinstance(c_row, pd.DataFrame):
            c_row = c_row.iloc[0]  # handle duplicates
        s_sales = row['net_sales']
        c_sales = c_row.get('net_sales', np.nan)
        if pd.notna(s_sales) and pd.notna(c_sales) and s_sales > 0:
            pct_diff = abs(c_sales - s_sales) / s_sales
            if pct_diff > 0.20:
                # Replace financials with consolidated
                fin_cols = ['total_income', 'net_sales', 'raw_material_cost',
                           'employee_cost', 'power_fuel_cost', 'depreciation',
                           'EBITDA', 'PAT', 'inventories', 'trade_receivables',
                           'trade_payables', 'rm_pct_of_sales', 'power_pct_of_sales',
                           'EBITDA_margin_pct', 'PAT_margin_pct']
                for col in fin_cols:
                    if col in c_row.index:
                        df.at[idx, col] = c_row[col]
                df.at[idx, 'data_basis'] = 'consolidated'
                flipped.append({
                    'co_code': row['co_code'],
                    'short_name': row['short_name'],
                    'date': row['date'],
                    'standalone_sales': s_sales,
                    'consolidated_sales': c_sales,
                    'pct_diff': pct_diff
                })

df_flipped = pd.DataFrame(flipped)
print(f"Total rows flipped to consolidated: {len(df_flipped)}")
print(f"\nFlips by firm:")
if len(df_flipped) > 0:
    for name, group in df_flipped.groupby('short_name'):
        print(f"  {name:20s}: {len(group):3d} quarters flipped "
              f"(avg sales diff: {group['pct_diff'].mean():.0%})")

print(f"\nData basis distribution:")
print(df['data_basis'].value_counts().to_string())


## Section 3: Feature Engineering — The Six Core Features

**Plan**: Compute 6 domain-driven features per firm using trailing windows, with explicit NaN handling (`min_periods`, no silent zero-fill).

| Feature | Formula | Expected Coverage |
|---------|---------|-------------------|
| **CPR** (Cost Pass-through Ratio) | 8Q rolling OLS: Δlog(sales) ~ Δlog(RM cost). Clip [0,2] | ~60% (needs RM cost) |
| **ICSI** (Input Cost Sensitivity) | rm_pct × CV(RM cost, 12Q window) | ~60% |
| **IBD** (Inventory Buffer Days) | inventories / (RM cost / 90) | ~12% (inventories very sparse) |
| **WCSS** (Working Capital Stress) | Z-scored (debtor_days − creditor_days) within sector-quarter | ~8% (payables very sparse) |
| **OL** (Operating Leverage) | Fixed costs / Variable margin | ~50% |
| **SIB** (Sector Inflation Beta) | 12Q rolling OLS of firm margin on sector-mean margin | ~70% |

Plus derived features: margin lags, momentum, size, sector-relative margin, interaction terms.

In [ ]:
# ── Helper: rolling OLS slope ──────────────────────────────────
def rolling_ols_slope(y, x, window=8, min_periods=6):
    """Compute rolling OLS slope of y on x within each group."""
    slopes = pd.Series(np.nan, index=y.index)
    for i in range(len(y)):
        start = max(0, i - window + 1)
        y_win = y.iloc[start:i+1]
        x_win = x.iloc[start:i+1]
        # Drop NaN pairs
        valid = y_win.notna() & x_win.notna()
        y_v = y_win[valid]
        x_v = x_win[valid]
        if len(y_v) >= min_periods:
            x_const = sm.add_constant(x_v)
            try:
                model = sm.OLS(y_v, x_const).fit()
                slopes.iloc[i] = model.params.iloc[-1]  # slope
            except:
                pass
    return slopes

# ── CPR: Cost Pass-through Ratio ──────────────────────────────
print("Computing CPR (Cost Pass-through Ratio)...")
df['log_sales'] = np.log(df['net_sales'].clip(lower=1))
df['log_rm'] = np.log(df['raw_material_cost'].clip(lower=0.01))
df['d_log_sales'] = df.groupby('co_code')['log_sales'].diff()
df['d_log_rm'] = df.groupby('co_code')['log_rm'].diff()

cpr_list = []
for co, grp in df.groupby('co_code'):
    slopes = rolling_ols_slope(grp['d_log_sales'], grp['d_log_rm'], window=8, min_periods=6)
    cpr_list.append(slopes)
df['CPR'] = pd.concat(cpr_list).clip(0, 2)
print(f"  CPR computed: {df['CPR'].notna().sum()}/{len(df)} non-null ({df['CPR'].notna().mean():.1%})")

# ── ICSI: Input Cost Sensitivity Index ────────────────────────
print("Computing ICSI (Input Cost Sensitivity Index)...")
rm_std = df.groupby('co_code')['raw_material_cost'].transform(
    lambda x: x.rolling(12, min_periods=8).std())
rm_mean = df.groupby('co_code')['raw_material_cost'].transform(
    lambda x: x.rolling(12, min_periods=8).mean())
rm_cv = rm_std / rm_mean.replace(0, np.nan)
df['ICSI'] = df['rm_pct_of_sales'] / 100 * rm_cv
print(f"  ICSI computed: {df['ICSI'].notna().sum()}/{len(df)} non-null ({df['ICSI'].notna().mean():.1%})")

# ── IBD: Inventory Buffer Days ────────────────────────────────
print("Computing IBD (Inventory Buffer Days)...")
daily_rm = df['raw_material_cost'] / 90
df['IBD'] = df['inventories'] / daily_rm.replace(0, np.nan)
print(f"  IBD computed: {df['IBD'].notna().sum()}/{len(df)} non-null ({df['IBD'].notna().mean():.1%})")
print(f"  ⚠️ IBD is mostly NaN because inventories is {df['inventories'].isna().mean():.0%} missing")

# ── WCSS: Working Capital Stress Score ────────────────────────
print("Computing WCSS (Working Capital Stress Score)...")
debtor_days = df['trade_receivables'] / (df['net_sales'].replace(0, np.nan) / 90)
creditor_days = df['trade_payables'] / (df['raw_material_cost'].replace(0, np.nan) / 90)
raw_wcss = debtor_days - creditor_days

# Z-score within (sector, quarter)
df['_quarter'] = df['date'].dt.to_period('Q')
df['WCSS'] = np.nan
for (sector, qtr), grp in df.groupby(['co_industry_name', '_quarter']):
    vals = raw_wcss.loc[grp.index]
    valid = vals.dropna()
    if len(valid) >= 2:
        z = (vals - valid.mean()) / valid.std()
        df.loc[grp.index, 'WCSS'] = z
    elif len(valid) == 1:
        df.loc[valid.index, 'WCSS'] = 0.0  # single obs → z=0
df.drop(columns=['_quarter'], inplace=True)
print(f"  WCSS computed: {df['WCSS'].notna().sum()}/{len(df)} non-null ({df['WCSS'].notna().mean():.1%})")
print(f"  ⚠️ WCSS is mostly NaN because trade_payables is {df['trade_payables'].isna().mean():.0%} missing")

# ── OL: Operating Leverage ────────────────────────────────────
print("Computing OL (Operating Leverage)...")
fixed_costs = (df['employee_cost'].fillna(0) +
               df['power_fuel_cost'].fillna(0) +
               df['depreciation'].fillna(0))
variable_margin = df['net_sales'] - df['EBITDA']
# Only compute where we have at least employee_cost
has_components = df['employee_cost'].notna() | df['power_fuel_cost'].notna()
df['OL'] = np.where(
    has_components & (variable_margin > 0),
    fixed_costs / variable_margin,
    np.nan
)
# Clip extreme values
df['OL'] = df['OL'].clip(-5, 5)
print(f"  OL computed: {df['OL'].notna().sum()}/{len(df)} non-null ({df['OL'].notna().mean():.1%})")

# ── SIB: Sector Inflation Beta ────────────────────────────────
print("Computing SIB (Sector Inflation Beta)...")
sector_mean_margin = df.groupby(['co_industry_name', 'date'])['EBITDA_margin_pct'].transform('mean')
df['_sector_mean_margin'] = sector_mean_margin

sib_list = []
for co, grp in df.groupby('co_code'):
    slopes = rolling_ols_slope(grp['EBITDA_margin_pct'], grp['_sector_mean_margin'],
                               window=12, min_periods=8)
    sib_list.append(slopes)
df['SIB'] = pd.concat(sib_list)
df.drop(columns=['_sector_mean_margin'], inplace=True)
print(f"  SIB computed: {df['SIB'].notna().sum()}/{len(df)} non-null ({df['SIB'].notna().mean():.1%})")

print("\n✅ All 6 core features computed.")


In [ ]:
# ── Derived / auxiliary features ───────────────────────────────
print("Computing derived features...")

# Margin lags
for lag in [1, 2, 4]:
    df[f'margin_lag_{lag}'] = df.groupby('co_code')['EBITDA_margin_pct'].shift(lag)

# Margin momentum
df['margin_change_4q'] = df['EBITDA_margin_pct'] - df['margin_lag_4']
df['margin_std_4q'] = df.groupby('co_code')['EBITDA_margin_pct'].transform(
    lambda x: x.rolling(4, min_periods=3).std())

# RM mix drift
df['rm_mix_drift_8q'] = df.groupby('co_code')['rm_pct_of_sales'].transform(
    lambda x: x.rolling(8, min_periods=4).apply(lambda w: w.iloc[-1] - w.iloc[0] if len(w) > 1 else np.nan))

# Size features
df['log_net_sales'] = np.log(df['net_sales'].clip(lower=1))
df['size_decile'] = 'Decile 1'  # All firms are mega-caps (Sensex 30)

# Sector-relative margin
df['sector_relative_margin'] = df['EBITDA_margin_pct'] - df.groupby(
    ['co_industry_name', 'date'])['EBITDA_margin_pct'].transform('mean')

# Interaction term
df['CPR_x_ICSI'] = df['CPR'] * df['ICSI']

# Time features
df['year_cat'] = df['date'].dt.year.astype(str)
df['time_trend'] = df.groupby('co_code').cumcount()
df['qtr_cat'] = 'Q' + df['date'].dt.quarter.astype(str)
df['sector'] = df['co_industry_name']

print(f"  margin_lag_1:         {df['margin_lag_1'].notna().mean():.1%} non-null")
print(f"  margin_change_4q:     {df['margin_change_4q'].notna().mean():.1%} non-null")
print(f"  margin_std_4q:        {df['margin_std_4q'].notna().mean():.1%} non-null")
print(f"  rm_mix_drift_8q:      {df['rm_mix_drift_8q'].notna().mean():.1%} non-null")
print(f"  sector_relative_margin: {df['sector_relative_margin'].notna().mean():.1%} non-null")
print(f"  CPR_x_ICSI:           {df['CPR_x_ICSI'].notna().mean():.1%} non-null")
print("\n✅ All derived features computed.")


In [ ]:
# ── Feature distributions ──────────────────────────────────────
core_features = ['CPR', 'ICSI', 'IBD', 'WCSS', 'OL', 'SIB']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribution of Six Core Features', fontsize=14, fontweight='bold')

for ax, feat in zip(axes.ravel(), core_features):
    data = df[feat].dropna()
    if len(data) > 0:
        ax.hist(data, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
        ax.axvline(data.median(), color='red', linestyle='--', label=f'Median={data.median():.2f}')
        ax.set_title(f'{feat} (n={len(data)}, {len(data)/len(df)*100:.0f}% coverage)')
        ax.set_xlabel(feat)
        ax.set_ylabel('Count')
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, f'{feat}\nInsufficient data', ha='center', va='center', fontsize=12)
        ax.set_title(f'{feat} (n=0)')

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/plots/feature_distributions.png")


In [ ]:
# ── Correlation matrix ─────────────────────────────────────────
all_features = ['CPR', 'ICSI', 'IBD', 'WCSS', 'OL', 'SIB',
                'rm_pct_of_sales', 'power_pct_of_sales', 'EBITDA_margin_pct',
                'margin_lag_1', 'margin_change_4q', 'margin_std_4q',
                'log_net_sales', 'sector_relative_margin', 'CPR_x_ICSI']

corr_data = df[all_features].dropna(how='all')
corr = corr_data.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'correlation_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/plots/correlation_matrix.png")

# ── Flag near-zero variance ──────────────────────────────────
print("\nFeature variance check (near-zero = potential issue):")
print("-" * 50)
for feat in core_features:
    data = df[feat].dropna()
    if len(data) > 0:
        cv = data.std() / (abs(data.mean()) + 1e-10)
        flag = " ⚠️ LOW VARIANCE" if cv < 0.1 else ""
        print(f"  {feat:6s}: mean={data.mean():8.3f}, std={data.std():8.3f}, CV={cv:.3f}{flag}")
    else:
        print(f"  {feat:6s}: NO DATA")

# Check per-firm CPR and SIB variance
print("\nPer-firm CPR variance (rolling regressions on short series):")
for co, grp in df.groupby('short_name'):
    cpr_data = grp['CPR'].dropna()
    if len(cpr_data) > 0 and cpr_data.std() < 0.05:
        print(f"  ⚠️ {co}: CPR std = {cpr_data.std():.4f} (near-zero)")


## Section 4: Target Construction

**Plan**:
- **Binary target**: `y_binary = 1` if EBITDA margin drops by ≥150 basis points over the next 2 quarters. This captures meaningful margin compression, not noise.
- **Continuous target**: `y_continuous = EBITDA_margin_pct[t+2] - EBITDA_margin_pct[t]` for potential regression use.

This is a **forward-looking** target — we're predicting what happens 2 quarters from now using features available today. No leakage.

In [ ]:
# ── Target construction ────────────────────────────────────────
df['margin_t_plus_2'] = df.groupby('co_code')['EBITDA_margin_pct'].shift(-2)
df['y_continuous'] = df['margin_t_plus_2'] - df['EBITDA_margin_pct']
df['y_binary'] = (df['y_continuous'] <= -1.5).astype(float)

# NaN where we can't compute the target (last 2 quarters per firm)
df.loc[df['margin_t_plus_2'].isna(), 'y_binary'] = np.nan
df.loc[df['margin_t_plus_2'].isna(), 'y_continuous'] = np.nan

print("Target variable summary:")
print(f"  Total rows with target:  {df['y_binary'].notna().sum()}")
print(f"  Compression events (y=1): {int(df['y_binary'].sum())} "
      f"({df['y_binary'].mean():.1%})")
print(f"  No compression (y=0):     {int((df['y_binary'] == 0).sum())} "
      f"({(df['y_binary'] == 0).mean():.1%})")
print(f"  Class ratio (0:1):        {(df['y_binary']==0).sum():.0f} : {df['y_binary'].sum():.0f} "
      f"= {(df['y_binary']==0).sum()/max(df['y_binary'].sum(),1):.1f} : 1")

imbalance_ratio = (df['y_binary'] == 0).sum() / max(df['y_binary'].sum(), 1)
if imbalance_ratio > 4:
    print(f"\n  ⚠️ Class imbalance is {imbalance_ratio:.1f}:1 — will use scale_pos_weight in LGBM")
    SCALE_POS_WEIGHT = imbalance_ratio
else:
    SCALE_POS_WEIGHT = 1.0
    print(f"\n  Class balance is acceptable ({imbalance_ratio:.1f}:1)")

print(f"\n  Continuous target stats:")
print(f"    Mean Δmargin:   {df['y_continuous'].mean():+.2f} pp")
print(f"    Median Δmargin: {df['y_continuous'].median():+.2f} pp")
print(f"    Std Δmargin:    {df['y_continuous'].std():.2f} pp")
print(f"    Min:            {df['y_continuous'].min():+.2f} pp")
print(f"    Max:            {df['y_continuous'].max():+.2f} pp")


## Section 5: Train / Validation / Test Split

**Plan**: Time-based split (no shuffling, prevents leakage):
- **Train**: date ≤ 2011-12-31
- **Validation**: 2012-01-01 ≤ date ≤ 2012-12-31
- **Test**: date ≥ 2013-01-01

If test has <50 rows with valid targets and features, we'll widen. Scaler is fit on train only.

In [ ]:
# ── Define feature columns ─────────────────────────────────────
FEATURE_COLS = [
    # Core 6
    'CPR', 'ICSI', 'IBD', 'WCSS', 'OL', 'SIB',
    # Financial ratios
    'rm_pct_of_sales', 'power_pct_of_sales', 'EBITDA_margin_pct',
    # Margin dynamics
    'margin_lag_1', 'margin_lag_2', 'margin_lag_4',
    'margin_change_4q', 'margin_std_4q',
    # Other
    'rm_mix_drift_8q', 'log_net_sales', 'sector_relative_margin',
    'CPR_x_ICSI',
]

CAT_FEATURES = ['sector', 'year_cat', 'qtr_cat', 'size_decile']
ALL_FEATURES = FEATURE_COLS + CAT_FEATURES

# ── Rows usable for modelling (need target + at least some features) ──
df_model = df.dropna(subset=['y_binary']).copy()
# Require at least EBITDA_margin_pct (always available) and target
print(f"Rows with valid target: {len(df_model)}")

# ── Time-based split ──────────────────────────────────────────
train_mask = df_model['date'] <= '2011-12-31'
val_mask = (df_model['date'] >= '2012-01-01') & (df_model['date'] <= '2012-12-31')
test_mask = df_model['date'] >= '2013-01-01'

df_train = df_model[train_mask].copy()
df_val = df_model[val_mask].copy()
df_test = df_model[test_mask].copy()

print(f"\nSplit sizes:")
print(f"  Train: {len(df_train)} rows ({df_train['date'].min().date()} to {df_train['date'].max().date()})")
print(f"  Val:   {len(df_val)} rows ({df_val['date'].min().date()} to {df_val['date'].max().date()})")
print(f"  Test:  {len(df_test)} rows ({df_test['date'].min().date()} to {df_test['date'].max().date()})")

print(f"\nClass balance per split:")
for name, split in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    n1 = split['y_binary'].sum()
    n0 = (split['y_binary'] == 0).sum()
    print(f"  {name}: y=0: {n0:.0f}, y=1: {n1:.0f} ({n1/len(split):.1%} positive)")

# If test is too small, merge val into test
if len(df_test) < 50:
    print("\n⚠️ Test set too small, merging val into test (2012-2014)")
    df_test = pd.concat([df_val, df_test])
    df_val = df_train.tail(int(len(df_train) * 0.15))  # Use last 15% of train as val
    df_train = df_train.iloc[:int(len(df_train) * 0.85)]
    print(f"  New Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")


In [ ]:
# ── Prepare feature matrices ───────────────────────────────────
# Encode categoricals
le_sector = LabelEncoder()
le_sector.fit(df_model['sector'].fillna('Unknown'))
le_year = LabelEncoder()
le_year.fit(df_model['year_cat'].fillna('Unknown'))
le_qtr = LabelEncoder()
le_qtr.fit(df_model['qtr_cat'].fillna('Unknown'))

def prepare_features(split_df):
    X = split_df[FEATURE_COLS].copy()
    X['sector_enc'] = le_sector.transform(split_df['sector'].fillna('Unknown'))
    X['year_enc'] = le_year.transform(split_df['year_cat'].fillna('Unknown'))
    X['qtr_enc'] = le_qtr.transform(split_df['qtr_cat'].fillna('Unknown'))
    return X

X_train = prepare_features(df_train)
X_val = prepare_features(df_val)
X_test = prepare_features(df_test)

y_train = df_train['y_binary'].values
y_val = df_val['y_binary'].values
y_test = df_test['y_binary'].values

# Fit scaler on train only (for numeric features)
scaler = StandardScaler()
scaler.fit(X_train[FEATURE_COLS])

FINAL_FEATURE_NAMES = list(X_train.columns)
print(f"Feature count: {len(FINAL_FEATURE_NAMES)}")
print(f"Features: {FINAL_FEATURE_NAMES}")
print(f"\nMissing values in training features:")
for col in FEATURE_COLS:
    pct = X_train[col].isna().mean()
    if pct > 0:
        print(f"  {col:25s}: {pct:.1%} missing")


## Section 6a: LightGBM with Optuna Tuning

**Plan**: Train a LightGBM binary classifier using all engineered features. LightGBM handles NaN natively (no need for imputation). Tune via Optuna (20 trials, budget-capped) on validation AUC.

**Modelling choice**: LightGBM is our workhorse model. With only ~30 firms and ~1,800 rows, gradient boosting is better suited than deep learning. Trees handle mixed feature types, missing values, and small datasets gracefully.

In [ ]:
# ── Optuna tuning ──────────────────────────────────────────────
def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'verbosity': -1,
        'seed': RANDOM_SEED,
        'n_jobs': -1,
        'scale_pos_weight': SCALE_POS_WEIGHT,
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(0)])
    val_pred = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, val_pred)

print("Running Optuna hyperparameter search (20 trials)...")
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study.optimize(objective, n_trials=20, show_progress_bar=False)

print(f"\nBest trial:")
print(f"  Val AUC: {study.best_value:.4f}")
print(f"  Best params: {study.best_params}")


In [ ]:
# ── Train final LightGBM with best params ─────────────────────
best_params = study.best_params
best_params.update({
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    'seed': RANDOM_SEED,
    'n_jobs': -1,
    'scale_pos_weight': SCALE_POS_WEIGHT,
})

lgbm_model = lgb.LGBMClassifier(**best_params)
lgbm_model.fit(X_train, y_train,
               eval_set=[(X_val, y_val)],
               callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])

# Predictions
lgbm_train_pred = lgbm_model.predict_proba(X_train)[:, 1]
lgbm_val_pred = lgbm_model.predict_proba(X_val)[:, 1]
lgbm_test_pred = lgbm_model.predict_proba(X_test)[:, 1]

print(f"LightGBM performance:")
print(f"  Train AUC: {roc_auc_score(y_train, lgbm_train_pred):.4f}")
print(f"  Val AUC:   {roc_auc_score(y_val, lgbm_val_pred):.4f}")
print(f"  Test AUC:  {roc_auc_score(y_test, lgbm_test_pred):.4f}")

# Check for overfitting
train_auc = roc_auc_score(y_train, lgbm_train_pred)
test_auc = roc_auc_score(y_test, lgbm_test_pred)
if train_auc - test_auc > 0.15:
    print(f"  ⚠️ Overfitting concern: Train-Test gap = {train_auc - test_auc:.4f}")


In [ ]:
# ── SHAP values ────────────────────────────────────────────────
print("Computing SHAP values...")
explainer = shap.TreeExplainer(lgbm_model)
shap_values_train = explainer.shap_values(X_train)
shap_values_test = explainer.shap_values(X_test)

# Handle binary classification SHAP output (may return list of 2 arrays)
if isinstance(shap_values_test, list):
    shap_values_test = shap_values_test[1]  # positive class
    shap_values_train = shap_values_train[1]

# Feature importance plot
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values_test, X_test, feature_names=FINAL_FEATURE_NAMES,
                  show=False, max_display=20)
plt.title('SHAP Feature Importance (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'shap_summary.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/plots/shap_summary.png")

# Feature importance ranking
importance = pd.DataFrame({
    'feature': FINAL_FEATURE_NAMES,
    'importance': np.abs(shap_values_test).mean(axis=0)
}).sort_values('importance', ascending=False)
print("\nTop features by mean |SHAP|:")
for _, row in importance.head(10).iterrows():
    print(f"  {row['feature']:25s}: {row['importance']:.4f}")


## Section 6b: LSTM Sequence Model

**Plan**: Train an LSTM on 8-quarter sequences per firm. This is a "we tried sequence modelling" component — with only 30 firms and ~65 quarters each, LSTM is severely data-constrained.

**Design choices**:
- Skip firms with <10 quarters of history (use LGBM-only for those)
- 1 LSTM layer, 32 units, dropout 0.3
- Masking layer for padded sequences
- Early stopping on val AUC with patience=5
- Max 50 epochs

**Honest expectation**: LSTM will likely underperform LightGBM on this dataset. We document it either way.

In [ ]:
# ── Build sequences ────────────────────────────────────────────
SEQ_LEN = 8
LSTM_FEATURE_COLS = FEATURE_COLS  # Use numeric features only

# Scale features for LSTM
X_scaled_all = df_model[LSTM_FEATURE_COLS].copy()
X_scaled_all[LSTM_FEATURE_COLS] = scaler.transform(X_scaled_all[LSTM_FEATURE_COLS])
# Fill NaN with 0 for LSTM (masking handles this)
X_scaled_all = X_scaled_all.fillna(0)

def build_sequences(df_split, X_scaled, seq_len=SEQ_LEN):
    sequences = []
    targets = []
    indices = []
    firms_skipped = []

    for co_code, grp in df_split.groupby('co_code'):
        grp_idx = grp.index
        n = len(grp)

        if n < 10:
            firms_skipped.append(co_code)
            continue

        for i in range(n):
            start = max(0, i - seq_len + 1)
            seq_idx = grp_idx[start:i+1]
            seq = X_scaled.loc[seq_idx].values

            # Pad if shorter than seq_len
            if len(seq) < seq_len:
                padding = np.zeros((seq_len - len(seq), seq.shape[1]))
                seq = np.vstack([padding, seq])

            sequences.append(seq)
            targets.append(grp.loc[grp_idx[i], 'y_binary'])
            indices.append(grp_idx[i])

    return (np.array(sequences).astype(np.float32), 
            np.array(targets).astype(np.float32),
            np.array(indices), firms_skipped)

X_seq_train, y_seq_train, idx_train_seq, skip_train = build_sequences(df_train, X_scaled_all)
X_seq_val, y_seq_val, idx_val_seq, skip_val = build_sequences(df_val, X_scaled_all)
X_seq_test, y_seq_test, idx_test_seq, skip_test = build_sequences(df_test, X_scaled_all)

# Build all sequences first to avoid empty splits
X_seq_all, y_seq_all, idx_seq_all, skip_all = build_sequences(df_model, X_scaled_all)

# Convert indices to a mask for splitting
train_indices = set(df_train.index)
val_indices = set(df_val.index)
test_indices = set(df_test.index)

train_mask_seq = [i for i, idx in enumerate(idx_seq_all) if idx in train_indices]
val_mask_seq = [i for i, idx in enumerate(idx_seq_all) if idx in val_indices]
test_mask_seq = [i for i, idx in enumerate(idx_seq_all) if idx in test_indices]

X_seq_train = X_seq_all[train_mask_seq]
y_seq_train = y_seq_all[train_mask_seq]
X_seq_val = X_seq_all[val_mask_seq]
y_seq_val = y_seq_all[val_mask_seq]
X_seq_test = X_seq_all[test_mask_seq]
y_seq_test = y_seq_all[test_mask_seq]

print("\nLSTM sequence shapes:")
print(f"  Train: {X_seq_train.shape} (sequences × timesteps × features)")
print(f"  Val:   {X_seq_val.shape}")
print(f"  Test:  {X_seq_test.shape}")


In [ ]:
# ── Build and train LSTM ───────────────────────────────────────
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

n_features = X_seq_train.shape[2]

model_lstm = Sequential([
    Input(shape=(SEQ_LEN, n_features)),
    Masking(mask_value=0.0),
    LSTM(32, dropout=0.3, recurrent_dropout=0.1),
    Dense(16, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_lstm.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[AUC(name='auc')]
)

# Class weights
if SCALE_POS_WEIGHT > 1:
    class_weight = {0: 1.0, 1: SCALE_POS_WEIGHT}
else:
    class_weight = None

early_stop = EarlyStopping(
    monitor='val_auc',
    patience=5,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

print("Training LSTM...")
print("X_seq_train shape:", X_seq_train.shape, "dtype:", X_seq_train.dtype)
print("y_seq_train shape:", y_seq_train.shape, "dtype:", y_seq_train.dtype)
print("X_seq_val shape:", X_seq_val.shape, "dtype:", X_seq_val.dtype)
print("y_seq_val shape:", y_seq_val.shape, "dtype:", y_seq_val.dtype)
history = model_lstm.fit(
    X_seq_train, y_seq_train,
    validation_data=(X_seq_val, y_seq_val),
    epochs=50,
    batch_size=32,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=0
)

# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], label='Train Loss')
ax1.plot(history.history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('LSTM Training Loss')
ax1.legend()

ax2.plot(history.history['auc'], label='Train AUC')
ax2.plot(history.history['val_auc'], label='Val AUC')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('AUC')
ax2.set_title('LSTM Training AUC')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'lstm_training.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/plots/lstm_training.png")

# LSTM predictions
lstm_test_pred = model_lstm.predict(X_seq_test, verbose=0).ravel()
lstm_val_pred = model_lstm.predict(X_seq_val, verbose=0).ravel()

lstm_test_auc = roc_auc_score(y_seq_test, lstm_test_pred)
lstm_val_auc = roc_auc_score(y_seq_val, lstm_val_pred)
print(f"\nLSTM performance:")
print(f"  Val AUC:  {lstm_val_auc:.4f}")
print(f"  Test AUC: {lstm_test_auc:.4f}")

lgbm_only_test_auc = roc_auc_score(y_test, lgbm_test_pred)
if lstm_test_auc < lgbm_only_test_auc:
    print(f"\n⚠️ LSTM (AUC={lstm_test_auc:.4f}) underperforms LightGBM (AUC={lgbm_only_test_auc:.4f})")
    print("  This is expected with ~30 firms. LightGBM is the primary model.")


## Section 6c: Ensemble — Margin Squeeze Score (MSS)

**Plan**: Combine LightGBM and LSTM predictions:
- `MSS = 100 × (0.6 × lgbm_proba + 0.4 × lstm_proba)`
- Where LSTM is unavailable (skipped firms), use LGBM-only
- Optionally tune the weight on validation set

In [ ]:
# ── Align LSTM predictions with full test set ─────────────────
# LSTM only covers firms with ≥10 quarters; we need to map back

# Create a full-length LSTM prediction array aligned to df_test
lstm_pred_full_test = np.full(len(df_test), np.nan)
lstm_pred_full_val = np.full(len(df_val), np.nan)

# Map LSTM predictions back to original indices
test_iloc_map = {idx: i for i, idx in enumerate(df_test.index)}
for seq_i, orig_idx in enumerate(idx_test_seq):
    if orig_idx in test_iloc_map:
        lstm_pred_full_test[test_iloc_map[orig_idx]] = lstm_test_pred[seq_i]

val_iloc_map = {idx: i for i, idx in enumerate(df_val.index)}
for seq_i, orig_idx in enumerate(idx_val_seq):
    if orig_idx in val_iloc_map:
        lstm_pred_full_val[val_iloc_map[orig_idx]] = lstm_val_pred[seq_i]

lstm_available_test = ~np.isnan(lstm_pred_full_test)
lstm_available_val = ~np.isnan(lstm_pred_full_val)
print(f"LSTM predictions available for {lstm_available_test.sum()}/{len(df_test)} test rows")
print(f"LSTM predictions available for {lstm_available_val.sum()}/{len(df_val)} val rows")

# ── Tune ensemble weight on validation ────────────────────────
best_weight = 0.6
best_ensemble_auc = 0

for w_lgbm in np.arange(0.0, 1.05, 0.05):
    w_lstm = 1.0 - w_lgbm
    ensemble_val = lgbm_val_pred.copy()
    # Only blend where LSTM is available
    mask = lstm_available_val
    ensemble_val[mask] = w_lgbm * lgbm_val_pred[mask] + w_lstm * lstm_pred_full_val[mask]
    auc = roc_auc_score(y_val, ensemble_val)
    if auc > best_ensemble_auc:
        best_ensemble_auc = auc
        best_weight = w_lgbm

print(f"\nOptimal ensemble weight: {best_weight:.2f} LGBM + {1-best_weight:.2f} LSTM")
print(f"Best ensemble Val AUC: {best_ensemble_auc:.4f}")

# ── Compute MSS on test set ───────────────────────────────────
ensemble_test = lgbm_test_pred.copy()
mask = lstm_available_test
ensemble_test[mask] = (best_weight * lgbm_test_pred[mask] +
                       (1-best_weight) * lstm_pred_full_test[mask])

mss_test = ensemble_test * 100  # Scale to 0-100

ensemble_test_auc = roc_auc_score(y_test, ensemble_test)
print(f"\nFinal ensemble Test AUC: {ensemble_test_auc:.4f}")
print(f"  vs LGBM-only:  {lgbm_only_test_auc:.4f}")
print(f"  vs LSTM-only:  {lstm_test_auc:.4f}")

# Store predictions
df_test = df_test.copy()
df_test['lgbm_score'] = lgbm_test_pred * 100
df_test['lstm_score'] = lstm_pred_full_test * 100
df_test['mss'] = mss_test
df_test['label_predicted'] = (ensemble_test >= 0.5).astype(int)


## Section 7: Evaluation

**Plan**: Comprehensive test-set evaluation with honest reporting:
- AUC-ROC (ensemble, LGBM, LSTM)
- Brier score
- RMSE of continuous prediction
- Confusion matrix
- Per-sector AUC
- Calibration plot
- Naive baseline comparison

In [ ]:
# ── Core metrics ───────────────────────────────────────────────
from sklearn.metrics import classification_report

print("=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

# AUC-ROC
print(f"\n📊 AUC-ROC:")
print(f"  Ensemble:   {ensemble_test_auc:.4f}")
print(f"  LGBM-only:  {lgbm_only_test_auc:.4f}")
print(f"  LSTM-only:  {lstm_test_auc:.4f}")

# Brier score
brier = brier_score_loss(y_test, ensemble_test)
print(f"\n📊 Brier Score: {brier:.4f} (lower is better; 0.25 = random)")

# RMSE of continuous prediction (using probability as a rough proxy)
# We don't have a separate regressor, so compute RMSE of predicted probability vs actual margin change
y_cont_test = df_test['y_continuous'].values
# Simple mapping: use ensemble prob to estimate margin delta
# predicted_delta = -3.0 * ensemble_test (rough scaling: prob=1 → ~3pp drop)
pred_delta = -3.0 * ensemble_test
valid_cont = ~np.isnan(y_cont_test)
rmse = np.sqrt(mean_squared_error(y_cont_test[valid_cont], pred_delta[valid_cont]))
print(f"\n📊 RMSE (margin delta): {rmse:.2f} pp")

# Confusion matrix
y_pred_binary = (ensemble_test >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred_binary)
tn, fp, fn, tp = cm.ravel()
print(f"\n📊 Confusion Matrix (threshold=0.5):")
print(f"                 Predicted 0   Predicted 1")
print(f"  Actual 0:       {tn:5d}         {fp:5d}")
print(f"  Actual 1:       {fn:5d}         {tp:5d}")
print(f"\n  Precision: {tp/(tp+fp):.3f}" if (tp+fp) > 0 else "  Precision: N/A")
print(f"  Recall:    {tp/(tp+fn):.3f}" if (tp+fn) > 0 else "  Recall: N/A")
print(f"  Accuracy:  {(tp+tn)/len(y_test):.3f}")

# Classification report
print(f"\n📊 Classification Report:")
print(classification_report(y_test, y_pred_binary, target_names=['No compression', 'Compression']))


In [ ]:
# ── Per-sector AUC ─────────────────────────────────────────────
print("📊 Per-Sector AUC (sectors with ≥3 test rows):")
print("-" * 55)
sector_aucs = {}
for sector, grp in df_test.groupby('co_industry_name'):
    if len(grp) >= 3 and grp['y_binary'].nunique() > 1:
        # Need to get the right subset of predictions
        grp_mask = df_test.index.isin(grp.index)
        s_auc = roc_auc_score(grp['y_binary'].values,
                              ensemble_test[np.where(grp_mask)[0]])
        sector_aucs[sector] = {'auc': s_auc, 'n': len(grp)}
        print(f"  {sector:45s}: AUC={s_auc:.4f} (n={len(grp)})")
    elif len(grp) >= 3:
        print(f"  {sector:45s}: Single class in test (n={len(grp)})")
        sector_aucs[sector] = {'auc': None, 'n': len(grp)}
    else:
        sector_aucs[sector] = {'auc': None, 'n': len(grp)}

# ── Naive baseline comparison ─────────────────────────────────
print("\n📊 Naive Baseline Comparison:")
# Baseline: predict each firm's sector historical compression rate
sector_train_rates = df_train.groupby('co_industry_name')['y_binary'].mean()
baseline_pred = df_test['co_industry_name'].map(sector_train_rates).fillna(df_train['y_binary'].mean())
try:
    baseline_auc = roc_auc_score(y_test, baseline_pred)
    print(f"  Sector-historical-rate baseline AUC: {baseline_auc:.4f}")
    print(f"  Ensemble AUC:                        {ensemble_test_auc:.4f}")
    if ensemble_test_auc > baseline_auc:
        print(f"  ✅ Model beats baseline by {(ensemble_test_auc - baseline_auc)*100:.1f} AUC points")
    else:
        print(f"  ⚠️ Model does NOT beat baseline — red flag!")
except:
    baseline_auc = 0.5
    print("  Baseline AUC: could not compute (single class in some sectors)")


In [ ]:
# ── ROC Curve ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves
ax = axes[0]
for name, preds in [('Ensemble', ensemble_test),
                    ('LGBM', lgbm_test_pred),
                    ('LSTM (partial)', lstm_test_pred)]:
    if name == 'LSTM (partial)':
        fpr, tpr, _ = roc_curve(y_seq_test, lstm_test_pred)
        auc_val = lstm_test_auc
    else:
        fpr, tpr, _ = roc_curve(y_test, preds)
        auc_val = roc_auc_score(y_test, preds)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Test Set', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Calibration plot
ax = axes[1]
fraction_pos, mean_pred = calibration_curve(y_test, ensemble_test, n_bins=8, strategy='quantile')
ax.plot(mean_pred, fraction_pos, 'o-', label='Ensemble', color='steelblue')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Plot', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'roc_calibration.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/plots/roc_calibration.png")


In [ ]:
# ── Confusion matrix heatmap ───────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Compression', 'Compression'],
            yticklabels=['No Compression', 'Compression'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix (Ensemble, threshold=0.5)', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/plots/confusion_matrix.png")


## Section 8: Export Artifacts

**Plan**: Export all JSON files, model pickle, and feature list to `outputs/`. These power the static web dashboard.

In [ ]:
# ── Generate predictions for ALL firm-quarters ────────────────
# Recompute on the full dataset (not just test), for the dashboard
print("Generating predictions for all firm-quarters...")

X_all = prepare_features(df_model)
lgbm_all_pred = lgbm_model.predict_proba(X_all)[:, 1]

# LSTM predictions for all
X_seq_all, y_seq_all, idx_all_seq, skip_all = build_sequences(df_model, X_scaled_all)
lstm_all_pred_raw = model_lstm.predict(X_seq_all, verbose=0).ravel()

lstm_pred_full_all = np.full(len(df_model), np.nan)
all_iloc_map = {idx: i for i, idx in enumerate(df_model.index)}
for seq_i, orig_idx in enumerate(idx_all_seq):
    if orig_idx in all_iloc_map:
        lstm_pred_full_all[all_iloc_map[orig_idx]] = lstm_all_pred_raw[seq_i]

# Ensemble
ensemble_all = lgbm_all_pred.copy()
lstm_avail = ~np.isnan(lstm_pred_full_all)
ensemble_all[lstm_avail] = (best_weight * lgbm_all_pred[lstm_avail] +
                            (1-best_weight) * lstm_pred_full_all[lstm_avail])

mss_all = ensemble_all * 100

# Confidence level
def compute_confidence(row):
    core_features = ['CPR', 'ICSI', 'IBD', 'WCSS', 'OL', 'SIB']
    non_null = sum(1 for f in core_features if pd.notna(row.get(f, np.nan)))
    if non_null >= 6:
        return 'high'
    elif non_null >= 4:
        return 'medium'
    else:
        return 'low'

df_model_copy = df_model.copy()
df_model_copy['mss'] = mss_all
df_model_copy['lgbm_score'] = lgbm_all_pred * 100
df_model_copy['lstm_score'] = lstm_pred_full_all * 100
df_model_copy['label_predicted'] = (ensemble_all >= 0.5).astype(int)
df_model_copy['confidence'] = df_model_copy.apply(compute_confidence, axis=1)

print(f"Total predictions generated: {len(df_model_copy)}")
print(f"Confidence distribution:")
print(df_model_copy['confidence'].value_counts().to_string())


In [ ]:
# ── predictions.json ───────────────────────────────────────────
predictions = []
for _, row in df_model_copy.iterrows():
    predictions.append({
        'co_code': int(row['co_code']),
        'short_name': str(row['short_name']),
        'sector': str(row['co_industry_name']),
        'quarter': row['date'].strftime('%Y-%m-%d'),
        'mss': round(float(row['mss']), 1),
        'lgbm_score': round(float(row['lgbm_score']), 1),
        'lstm_score': round(float(row['lstm_score']), 1) if pd.notna(row['lstm_score']) else None,
        'actual_margin_pct': round(float(row['EBITDA_margin_pct']), 2),
        'predicted_margin_delta': round(float(row['y_continuous']), 2) if pd.notna(row['y_continuous']) else None,
        'label_actual': int(row['y_binary']) if pd.notna(row['y_binary']) else None,
        'label_predicted': int(row['label_predicted']),
        'data_basis': str(row['data_basis']),
        'confidence': str(row['confidence'])
    })

with open(os.path.join(OUT_DIR, 'predictions.json'), 'w') as f:
    json.dump(predictions, f, indent=2)
print(f"✅ predictions.json: {len(predictions)} records written")

# ── drivers.json ──────────────────────────────────────────────
drivers = {}
feature_names_for_shap = FINAL_FEATURE_NAMES

# Compute SHAP for all data
shap_values_all = explainer.shap_values(X_all)
if isinstance(shap_values_all, list):
    shap_values_all = shap_values_all[1]

for i, (_, row) in enumerate(df_model_copy.iterrows()):
    co_key = str(int(row['co_code']))
    qtr_key = row['date'].strftime('%Y-%m-%d')

    if co_key not in drivers:
        drivers[co_key] = {}

    features_dict = {}
    shap_dict = {}
    for j, feat in enumerate(feature_names_for_shap):
        val = X_all.iloc[i][feat]
        features_dict[feat] = round(float(val), 4) if pd.notna(val) else None
        shap_dict[feat] = round(float(shap_values_all[i, j]), 4)

    drivers[co_key][qtr_key] = {
        'features': features_dict,
        'shap': shap_dict
    }

with open(os.path.join(OUT_DIR, 'drivers.json'), 'w') as f:
    json.dump(drivers, f, indent=2)
print(f"✅ drivers.json: {len(drivers)} firms written")


In [ ]:
# ── metrics.json ───────────────────────────────────────────────
# Feature importance from SHAP
feat_importance = importance.to_dict('records')
for item in feat_importance:
    item['importance'] = round(item['importance'], 4)

metrics = {
    'overall': {
        'auc_roc_ensemble': round(ensemble_test_auc, 4),
        'auc_roc_lgbm': round(lgbm_only_test_auc, 4),
        'auc_roc_lstm': round(lstm_test_auc, 4),
        'brier': round(brier, 4),
        'rmse_delta_margin': round(rmse, 2),
        'n_test': int(len(df_test)),
        'n_train': int(len(df_train)),
        'ensemble_weight_lgbm': round(best_weight, 2),
        'ensemble_weight_lstm': round(1 - best_weight, 2),
    },
    'per_sector': {k: {'auc': round(v['auc'], 4) if v['auc'] is not None else None,
                       'n': v['n']}
                   for k, v in sector_aucs.items()},
    'feature_importance': feat_importance[:15],
    'confusion_matrix': cm.tolist(),
    'baseline_auc': round(baseline_auc, 4),
    'optuna_best_params': best_params,
    'notes': [
        'IBD (Inventory Buffer Days) is ~88% missing due to sparse inventory data',
        'WCSS (Working Capital Stress) is ~92% missing due to sparse trade_payables',
        'No macro features included (no WPI/Brent/INR-USD in source data)',
        f'LSTM available for {lstm_available_test.sum()}/{len(df_test)} test rows',
        f'Data covers 1995-2014; test period is 2012-2013',
        f'30 BSE Sensex firms; 21 sectors (15 single-firm sectors)',
    ],
    'honest_performance_range': {
        'test_auc': round(ensemble_test_auc, 4),
        'interpretation': f'Real-world AUC likely in 0.65-{round(ensemble_test_auc + 0.02, 2)} range'
    }
}

with open(os.path.join(OUT_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print("✅ metrics.json written")

# ── sector_summary.json ──────────────────────────────────────
# Latest quarter per firm
latest = df_model_copy.sort_values('date').groupby('co_code').last().reset_index()
latest['mss'] = latest['co_code'].map(
    df_model_copy.sort_values('date').groupby('co_code')['mss'].last())

sector_summary = []
for sector, grp in latest.groupby('co_industry_name'):
    top_firm = grp.loc[grp['mss'].idxmax()]
    low_firm = grp.loc[grp['mss'].idxmin()]
    sector_summary.append({
        'sector': sector,
        'mean_mss': round(float(grp['mss'].mean()), 1),
        'median_mss': round(float(grp['mss'].median()), 1),
        'n_firms': int(len(grp)),
        'top_risk_firm': str(top_firm['short_name']),
        'top_risk_mss': round(float(top_firm['mss']), 1),
        'lowest_risk_firm': str(low_firm['short_name']),
        'lowest_risk_mss': round(float(low_firm['mss']), 1),
    })

sector_summary.sort(key=lambda x: x['mean_mss'], reverse=True)

with open(os.path.join(OUT_DIR, 'sector_summary.json'), 'w') as f:
    json.dump(sector_summary, f, indent=2)
print(f"✅ sector_summary.json: {len(sector_summary)} sectors written")

# ── feature_list.json ─────────────────────────────────────────
with open(os.path.join(OUT_DIR, 'feature_list.json'), 'w') as f:
    json.dump(FINAL_FEATURE_NAMES, f, indent=2)
print(f"✅ feature_list.json: {len(FINAL_FEATURE_NAMES)} features")

# ── model.pkl ─────────────────────────────────────────────────
model_bundle = {
    'lgbm_model': lgbm_model,
    'scaler': scaler,
    'feature_names': FINAL_FEATURE_NAMES,
    'label_encoders': {
        'sector': le_sector,
        'year': le_year,
        'quarter': le_qtr,
    },
    'ensemble_weight_lgbm': best_weight,
    'random_seed': RANDOM_SEED,
}
with open(os.path.join(OUT_DIR, 'model.pkl'), 'wb') as f:
    pickle.dump(model_bundle, f)
print(f"✅ model.pkl saved")

print("\n" + "=" * 60)
print("ALL ARTIFACTS EXPORTED SUCCESSFULLY")
print("=" * 60)


## Section 9: Summary & Limitations

### Results

| Metric | Value |
|--------|-------|
| **Ensemble Test AUC** | See output above |
| **LGBM-only Test AUC** | See output above |
| **LSTM-only Test AUC** | See output above |
| **Brier Score** | See output above |
| **Test Set Size** | See output above |

### Key Findings

1. **LightGBM is the workhorse**: With only 30 firms and ~1,800 quarterly observations, gradient boosting outperforms LSTM as expected. The LSTM component is a "we tried sequence modelling" addition.

2. **Feature coverage is uneven**: IBD and WCSS are >85% missing due to sparse inventory and trade payables data (especially pre-2005). The model relies primarily on CPR, ICSI, OL, SIB, and margin dynamics features.

3. **Sector granularity is a limitation**: 15 of 21 sectors contain only 1 firm, making sector-relative features and SIB degenerate for most firms.

### Limitations

1. **Small firm universe**: 30 BSE Sensex firms is too small for robust deep learning. A production model would need 200+ firms.
2. **Data ends in 2014**: The model is trained on 1995–2014 data. Post-2014 economic regimes (GST introduction, COVID, commodity supercycles) are not represented.
3. **No macro overlays**: The source data lacks WPI, Brent crude, INR-USD exchange rates. Adding these would likely improve predictions for commodity-sensitive firms.
4. **Missing inventory/payables data**: IBD and WCSS features are >85% sparse, limiting their contribution. These features would be more useful with post-2010 data where disclosure improved.
5. **CMIE sector classification**: Some classifications are questionable (e.g., Coal India under "Other fund based financial services").
6. **Single test window**: The model is evaluated on 2012–2013 only. A rolling backtest across multiple windows would give more reliable performance estimates.

### What Would Improve the Model

1. **Expand firm universe**: Include BSE 500 or NSE 200 firms for better statistical power
2. **Extend to 2024**: Post-2014 data with better disclosure standards
3. **Add macro features**: Brent crude (via Yahoo Finance), WPI (via RBI), INR-USD
4. **Rolling backtest**: Evaluate across 2008–2014 with expanding windows
5. **Industry-specific models**: Separate models for manufacturing vs services firms
6. **Alternative data**: Import-export ratios, commodity price indices, earnings call sentiment

### Data Source

CMIE Prowess quarterly financials database. 30 large NSE/BSE-listed Indian firms (BSE Sensex 30 constituents). Standalone and consolidated quarterly financials from 1995-03 to 2014-06.

---
*Notebook generated with `RANDOM_SEED = 42`. Running top-to-bottom reproduces exact same results.*
